# Obtención y preparación de datos
**Proyecto:** Turismo y estructura empresarial en Medellín  

Este notebook carga las tres bases de datos originales directamente desde el repositorio de GitHub, aplica los filtros necesarios y construye la tabla unificada que se usará en el análisis.

---

**Bases de datos:**
- `BD1` — Estructura empresarial de Medellín según comunas y actividad económica *(Cámara de Comercio de Medellín / datos.gov.co)*
- `BD2` — Extranjeros no residentes que ingresan a Colombia *(MinCIT)*
- `BD3` — Llegada de pasajeros en vuelos nacionales *(MinCIT)*

## 1. Importar librerías

In [ ]:
import pandas as pd

## 2. Cargar las bases de datos originales desde GitHub

Los archivos están almacenados en el repositorio del proyecto. Se leen directamente desde la URL pública de GitHub usando pandas.

In [ ]:
# URL base del repositorio (reemplazar con la URL real del repositorio)
BASE_URL = "https://raw.githubusercontent.com/USUARIO/REPOSITORIO/main/datos/"

# BD1: Estructura empresarial por comunas y actividad económica
bd1 = pd.read_excel(BASE_URL + "BD1_estructura_empresarial.xlsx")

# BD2: Extranjeros no residentes
bd2 = pd.read_excel(BASE_URL + "BD2_extranjeros_no_residentes.xlsx")

# BD3: Pasajeros vuelos nacionales
bd3 = pd.read_excel(BASE_URL + "BD3_pasajeros_vuelos_nacionales.xlsx")

print("Bases de datos cargadas correctamente")

## 3. Exploración inicial de las bases de datos

Se revisa la estructura de cada base para entender sus columnas y el tipo de datos que contiene.

In [ ]:
print("=== BD1: Estructura empresarial ===")
print(bd1.shape)
bd1.head()

In [ ]:
print("=== BD2: Extranjeros no residentes ===")
print(bd2.shape)
bd2.head()

In [ ]:
print("=== BD3: Pasajeros vuelos nacionales ===")
print(bd3.shape)
bd3.head()

## 4. Preparación de BD1 — Estructura empresarial

Se filtran únicamente los códigos CIIU de interés para el proyecto y se conservan todas las comunas.

In [ ]:
# Códigos CIIU de interés para el análisis
ciiu_interes = [4711, 4721, 4723, 4773, 5519, 5611, 5619, 6615, 6810, 7912]

# Filtrar solo las filas con los códigos CIIU seleccionados
bd1_filtrada = bd1[bd1["CIIU"].isin(ciiu_interes)].copy()

# Reiniciar el índice
bd1_filtrada = bd1_filtrada.reset_index(drop=True)

print("Filas después del filtro:", len(bd1_filtrada))
bd1_filtrada.head()

In [ ]:
# Columnas de comunas que se van a conservar
comunas = [
    "Altavista", "Aranjuez", "Belén", "Buenos Aires", "Castilla",
    "Doce de Octubre", "El Poblado", "Guayabal", "La América",
    "La Candelaria", "Laureles-Estadio", "Manrique", "Palmitas",
    "Popular", "Robledo", "San Antonio de Prado", "San Cristóbal",
    "San Javier", "Santa Cruz", "Santa Elena", "Villa Hermosa",
    "Singeorreferenciar"
]

# Calcular el total de establecimientos por fila (sumando todas las comunas)
bd1_filtrada["Total_establecimientos"] = bd1_filtrada[comunas].sum(axis=1)

# Vista final de BD1 preparada
bd1_filtrada[["Año", "CIIU", "Descripción", "Total_establecimientos"] + comunas].head()

## 5. Preparación de BD2 — Extranjeros no residentes

Los datos ya vienen filtrados por Medellín. Se agrupa por año para obtener el total anual de visitantes extranjeros.

In [ ]:
# Verificar que los datos corresponden a Medellín
print("Ciudades en BD2:", bd2["Ciudad"].unique())

In [ ]:
# Agrupar por año: sumar todos los viajeros extranjeros por año
bd2_anual = bd2.groupby("Año")["Viajeros"].sum().reset_index()

# Renombrar columna para que sea más descriptiva
bd2_anual = bd2_anual.rename(columns={"Viajeros": "Viajeros_extranjeros"})

bd2_anual

## 6. Preparación de BD3 — Pasajeros vuelos nacionales

Los datos ya vienen filtrados por Medellín. Se agrupa por año para obtener el total anual de pasajeros nacionales.

In [ ]:
# Verificar que los datos corresponden a Medellín
print("Ciudades en BD3:", bd3["Ciudad"].unique())

In [ ]:
# Agrupar por año: sumar todos los pasajeros nacionales por año
bd3_anual = bd3.groupby("Año")["Pasajeros regulares"].sum().reset_index()

# Renombrar columna para que sea más descriptiva
bd3_anual = bd3_anual.rename(columns={"Pasajeros regulares": "Pasajeros_nacionales"})

bd3_anual

## 7. Unificación de BD2 y BD3 — Tabla de turismo por año

Se combinan los viajeros extranjeros y los pasajeros nacionales en una sola tabla anual.

In [ ]:
# Unir BD2 y BD3 por la columna Año
turismo_anual = pd.merge(bd2_anual, bd3_anual, on="Año", how="outer")

# Calcular el total de turistas (extranjeros + nacionales)
turismo_anual["Total_turistas"] = turismo_anual["Viajeros_extranjeros"] + turismo_anual["Pasajeros_nacionales"]

# Ordenar por año
turismo_anual = turismo_anual.sort_values("Año").reset_index(drop=True)

turismo_anual

## 8. Tabla final unificada — Año, turismo y estructura empresarial

Se cruza la tabla de turismo con la de estructura empresarial por año. La tabla final contiene: año, total de turistas, viajeros extranjeros, pasajeros nacionales, código CIIU, descripción de actividad y número de establecimientos por comuna.

In [ ]:
# Seleccionar columnas relevantes de BD1
bd1_para_unir = bd1_filtrada[["Año", "CIIU", "Descripción", "Total_establecimientos"] + comunas].copy()

# Unir con la tabla de turismo por año
tabla_final = pd.merge(bd1_para_unir, turismo_anual, on="Año", how="left")

# Ordenar por año y CIIU
tabla_final = tabla_final.sort_values(["Año", "CIIU"]).reset_index(drop=True)

print("Dimensiones de la tabla final:", tabla_final.shape)
tabla_final.head(20)

## 9. Guardar la tabla final

Se exporta la tabla unificada en formato `.xlsx` para su uso en el análisis.

In [ ]:
tabla_final.to_excel("tabla_final_unificada.xlsx", index=False)

print("Archivo guardado: tabla_final_unificada.xlsx")
print("Filas:", len(tabla_final))
print("Columnas:", list(tabla_final.columns))